<a href="https://colab.research.google.com/github/Su-3214/Dev_Pikofre/blob/master/SD4AI%E6%BC%94%E7%BF%921224%E6%A1%9D%E5%B4%8E%E9%81%94%E4%B9%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# == 1. 必要なライブラリのインストール ==
# Tavilyのバージョンを0.5.0に指定してインストールします
!pip install -U -q langchain langchain-google-genai langchain-community tavily-python==0.5.0

In [7]:
# == 2. APIキーの設定 ==
import os
from google.colab import userdata

# Tavilyの鍵は環境変数として設定します
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
# Google(Gemini)の鍵は変数として持っておきます
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

print("鍵のセットアップが完了しました！")

鍵のセットアップが完了しました！


In [8]:
# == 3. 各パーツの準備 ==
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.retrievers import TavilySearchAPIRetriever

# AIへの指示書（プロンプト）
prompt = ChatPromptTemplate.from_template("""
以下の検索した情報だけを踏まえて、友達のように親しみやすい言葉で質問に回答してください。

検索した情報:
{context}

ユーザーの質問:{question}
""")

# 検索エンジン（Retriever）の準備。上位5件の情報を拾ってきます
retriever = TavilySearchAPIRetriever(k=5)

# AIモデル（Gemini）の準備
# 💡 もしサーバー不調(503エラーなど)で動かない場合は、 model="gemini-2.5-flash" に書き換えてみてください。
model = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0,  # 0にすると、真面目にブレない回答をしてくれます
    google_api_key=GOOGLE_API_KEY
)

print("🧠 AIと検索エンジンの準備ができました！")

🧠 AIと検索エンジンの準備ができました！


In [9]:
# == 4. 処理の組み立て ==
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# パイプライン（チェイン）の作成
chain = (
    RunnableParallel(
        {
            # ユーザーの質問をそのまま保持して次に渡します
            "question": RunnablePassthrough(),
            # 質問をもとにWeb検索を実行します
            "context": retriever,
        }
    )
    # 検索結果と質問をもとに、AIに回答（answer）を作らせて追加します
    | RunnablePassthrough.assign(answer=(prompt | model | StrOutputParser()))
    # 最終的に「検索した情報」と「AIの回答」だけをピックアップして表示します
    .pick(["context", "answer"])
)

print("🔗 処理の流れ（チェイン）が完成しました！")

🔗 処理の流れ（チェイン）が完成しました！


In [10]:
# == 5. アシスタントへの質問実行 ==

# 💡 ここを好きな質問に変えてみましょう！
my_question = "小倉駅の、おすすめの食べ物と、博多駅からの行き方を教えて"

print(f"質問: {my_question}\n")
print("AIが一生懸命検索して考えています...\n")

# チェインを実行して答えをもらう
result = chain.invoke(my_question)

# 結果を見やすく表示する
print("✨ --- AIの回答 --- ✨")
print(result["answer"])
print("\n📚 --- 参考にした情報源（Context）--- 📚")
# どんなWebサイトを見たか確認してみよう
for doc in result["context"]:
    print(f"- {doc.page_content[:100]}...")

質問: 小倉駅の、おすすめの食べ物と、博多駅からの行き方を教えて

AIが一生懸命検索して考えています...

✨ --- AIの回答 --- ✨
博多駅から小倉駅への行き方は、実はすっごく簡単だよ！

* **新幹線で行く場合**：なんと**約15分**で着いちゃうよ！あっという間だから、時間を節約したいときにぴったり。
* **在来線で行く場合**：のんびり行くならこっち。**約50分**で到着するよ。

---

そして、小倉駅周辺で絶対に食べてほしいおすすめのグルメを紹介するね！

### 1. 小倉といえば！大人気の「うどん」
小倉はうどんがとっても有名なんだ。特におすすめなのがこの3つ！
* **駅ホームの立ち食いうどん（ぷらっとぴっと）**
  小倉駅のホーム（1・2番線、7・8番線）にある立ち食いうどん屋さん。ここの「かしわうどん」（500円）は、甘辛く煮た鶏肉（かしわ）の旨みがスープに溶け込んで、最後の一滴まで本当に美味しいの！
* **資さん（すけさん）うどん 魚町店**
  北九州発祥の超有名チェーン店で、駅から近い商店街にあるよ。人気No.1の「肉ごぼ天うどん」は、甘辛い牛肉ともっちり麺、サクサクのごぼ天の組み合わせが最高！
* **小倉名物の「肉うどん」**
  ゴロゴロ入った牛すじ肉に、たっぷりのおろし生姜を合わせて食べるのが小倉流。体がポカポカ温まるよ。

### 2. 実は小倉が発祥！「焼うどん」
終戦直後に焼きそばの代わりとして干しうどんで作ったのが始まりなんだって。
* 元祖の味を楽しめる「だるま堂」の、卵がのった「天窓」というメニューや、「お好み焼きいしん」の創作焼うどんがおすすめだよ。

### 3. とろ〜り練乳がたまらない「サニーパン」
パン好きなら絶対に外せないのがこれ！外はパリッと、中はふわふわのパンの中に、練乳がしたたるほどたっぷり入っていて、一度食べたらクセになる美味しさだよ。

### 4. 小倉の伝統料理「ぬか炊き」
サバやイワシなどの青魚を、ぬか床でコトコト煮込んだ小倉独自の郷土料理。味がしっかり染みていて、ご飯がとにかく進むよ！

### 5. ちょっと贅沢するなら…
* **田舎庵 小倉本店**：蒸さずにじっくり焼き上げる鰻（うなぎ）の名店。皮はパリッと、中はふっくらしていて絶品だよ。
* **稚加榮（ち